# Homework 5 — Monitoring

LLM Zoomcamp 2026 · Cohort

The module lessons build a monitoring stack by hand: a metrics dataclass, PostgreSQL storage, Streamlit and Grafana dashboards. This homework instruments the same course RAG with OpenTelemetry (OTel) directly instead: traces, spans, span attributes, and a custom SQLite exporter in place of Postgres.

**Knowledge base:** same 72 lesson pages from commit `8c1834d`, indexed with `minsearch.Index` (text search, no chunking). Same HW1 setup, reused as-is through the course's starter files.

## Setup

We use the course's official starter (`starter.py`, `rag_helper.py`), copied into this folder unchanged.

New dependencies for this module:

```bash
uv add opentelemetry-api opentelemetry-sdk
```

`opentelemetry-api` is the interface we import (`trace`, `Tracer`, `Span`). `opentelemetry-sdk` is the implementation that actually builds and processes spans.

In [1]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
from utils import load_openai_api_key

load_dotenv()
load_openai_api_key()

from starter import index, client

print("Index and OpenAI client ready.")

Index and OpenAI client ready.


## OpenTelemetry concepts

A few terms before writing any code:

- A trace is the end-to-end story of one request. For us, that's one call to `rag()`.
- A span is one operation inside a trace. A trace is a tree of spans, each with a name, a start/end time, and attributes. Our trace has three spans: `rag` (the whole call) and its two children, `search` and `llm`.
- Attributes are key-value pairs attached to a span for anything worth recording: token counts, cost, whatever we care about.
- An exporter decides where a finished span goes once its `with` block exits: console, a file, a database, a remote collector. It plugs into a span processor, which is registered on a `TracerProvider`.

Normally you'd call `trace.set_tracer_provider(provider)` once at startup, so any later `trace.get_tracer(...)` call picks up that provider automatically. We swap exporters a couple of times in this notebook, first console then SQLite, and OTel only allows the global provider to be set once per process. A second call is just a silent no-op. So instead we keep each `TracerProvider` as an explicit local object and call `provider.get_tracer(...)` directly. Same effect, without fighting the global singleton.

## Q1 & Q2 — First trace, and token/cost attributes

We build a `RAGTraced` subclass of `RAGBase` that wraps `rag()`, `search()`, and `llm()` each in their own span. Since `RAGBase.rag()` internally calls `self.search(...)` and `self.llm(...)`, and those are the overridden (traced) versions, the spans nest automatically: `rag` is current when `search` and `llm` start, so the SDK attaches them as its children without us passing a parent span by hand.

We fold Q2 in from the start. `llm()` already gets the raw `response` object back from `RAGBase.llm()`, not `.output_text` (that only gets extracted later, in `rag()`), so we can read `response.usage` and set `input_tokens` / `output_tokens` / `cost` on the `llm` span before returning it.

Cost uses the same per-token pricing we used in HW4's `evaluation_utils.calc_price` for `gpt-5.4-mini`: $0.75 / 1M input tokens, $4.50 / 1M output tokens.

For the exporter, we register two processors on one provider: `ConsoleSpanExporter`, to literally see each finished span printed (this is what Q1 asks us to count), and `InMemorySpanExporter`, so we can also read span timings programmatically later for Q3 instead of eyeballing timestamps in printed text.

In [2]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from rag_helper import RAGBase

INPUT_PRICE_PER_MILLION = 0.75   # gpt-5.4-mini, same rate as hw4-evaluation/evaluation_utils.py
OUTPUT_PRICE_PER_MILLION = 4.50


def calc_cost(usage):
    input_cost = (usage.input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (usage.output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    return input_cost + output_cost


console_provider = TracerProvider()
console_provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))

in_memory_exporter = InMemorySpanExporter()
console_provider.add_span_processor(SimpleSpanProcessor(in_memory_exporter))

tracer = console_provider.get_tracer("llm-zoomcamp")


class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            span.set_attribute("cost", calc_cost(usage))
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)


rag_traced = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xefc6ab925c7317ac01f44bc08703ff7b",
        "span_id": "0x31aac4c8ae3770a2",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe9c986cb50a916d2",
    "start_time": "2026-07-20T02:40:38.517842Z",
    "end_time": "2026-07-20T02:40:38.523145Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "01e99679-ed41-4bdd-8031-41bd3ca4c981",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xefc6ab925c7317ac01f44bc08703ff7b",
        "span_id": "0xd231d6dbb51c8ed2",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [3]:
finished_spans = in_memory_exporter.get_finished_spans()
print(f"Spans in this trace: {len(finished_spans)}")
for s in finished_spans:
    print(f"  - {s.name}")

Spans in this trace: 3
  - search
  - llm
  - rag


**Answer Q1: `3`**

One `rag()` call produces exactly 3 spans: `rag` (the outer call), `search`, and `llm`. They print to the console as three separate `ReadableSpan` entries, and the in-memory exporter's count above confirms it.

In [4]:
llm_span = next(s for s in finished_spans if s.name == "llm")
print("input_tokens:", llm_span.attributes["input_tokens"])
print("output_tokens:", llm_span.attributes["output_tokens"])
print("cost: $%.6f" % llm_span.attributes["cost"])

input_tokens: 7111
output_tokens: 105
cost: $0.005806


**Answer Q2: `7000`**

We observed 7111 input tokens for this query, closest to the `7000` option. That's mostly context: five retrieved lesson chunks plus the instructions and the question itself.

## Q3 — Span timing

Every span records its own `start_time`/`end_time` independently, so we can compare `search` and `llm` without touching the outer `rag` span. We read these straight from the spans captured by `InMemorySpanExporter` above.

In [4]:
for s in finished_spans:
    duration_ms = (s.end_time - s.start_time) / 1_000_000
    print(f"{s.name:>8}: {duration_ms:8.2f} ms")

  search:     5.30 ms
     llm:  4188.93 ms
     rag:  4195.48 ms


**Answer Q3: `Over 2000ms`**

`search` finishes in a couple of milliseconds since it's an in-memory `minsearch` lookup with no network involved. `llm` takes several seconds because it's a real HTTP call to the OpenAI API. Across repeated runs the `llm` span consistently landed well past the 2000ms mark.

## Q4 — Saving traces to SQLite

Now we replace the exporter with a custom one that writes finished spans to SQLite instead of printing them. Nothing about `RAGTraced` changes: same spans, same attributes, just a different destination. `SimpleSpanProcessor` stays synchronous, so each `rag()` call's rows are committed before the call returns.

We build a fresh `TracerProvider` for this (see the note in the OpenTelemetry concepts section above about why we don't reuse the global registration), delete any stale `traces.db` from a previous run, and reuse the same `RAGTraced` class. Only the tracer object it closes over changes.

In [5]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self, timeout_millis=30000):
        return True

In [6]:
import os

DB_PATH = "traces.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

sqlite_provider = TracerProvider()
sqlite_provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter(DB_PATH)))
tracer = sqlite_provider.get_tracer("llm-zoomcamp")

rag_sqlite = RAGTraced(index=index, llm_client=client)
rag_sqlite.rag(query)
print("done")

done


In [7]:
conn = sqlite3.connect(DB_PATH)
rows = conn.execute("SELECT name, start_time, end_time, input_tokens, output_tokens, cost FROM spans").fetchall()
for row in rows:
    print(row)

('search', 1784515931831585000, 1784515931834659000, None, None, None)
('llm', 1784515931836055000, 1784515934392981000, 7111, 98, 0.00577425)
('rag', 1784515931831550000, 1784515934394277000, None, None, None)


**Answer Q4: `rag`, `search`, and `llm`**

All three spans show up as rows in the `spans` table, with `input_tokens`/`output_tokens`/`cost` populated on the `llm` row and left `NULL` on `rag`/`search` (they never got those attributes).

## Q5 — Querying trace data

The `rag` span's duration includes both its children, so comparing it against `search`/`llm` would be comparing a total to its own parts. We exclude it and group by span name to see where time actually goes.

We run one more query through the same SQLite-backed `RAGTraced` instance first, so the table has more than one trace to aggregate over.

In [8]:
rag_sqlite.rag("What is the difference between hybrid search and vector search?")
print("second call done")

second call done


In [9]:
import pandas as pd

df = pd.read_sql("SELECT * FROM spans", conn)

duration_by_name = (
    df[df["name"] != "rag"]
    .assign(duration_ms=lambda d: (d["end_time"] - d["start_time"]) / 1_000_000)
    .groupby("name")["duration_ms"]
    .sum()
    .sort_values(ascending=False)
)
duration_by_name

name
llm       4607.257
search       8.252
Name: duration_ms, dtype: float64

**Answer Q5: `llm`**

Across both calls, `llm` accounts for essentially all the measured time: a few seconds total, against a few milliseconds for `search`. The LLM call, not retrieval, is where the latency budget goes.

## Q6 — Token stability across runs

We want to know whether the same query always retrieves the same context. If `input_tokens` stays constant across repeated calls, retrieval is deterministic; if it swings, something in `search` is unstable.

Q4/Q5 mixed two different queries in `traces.db`, which isn't a fair comparison for this question, so we reset the database and run the exact Q1 query four times against a fresh `RAGTraced` instance.

In [10]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

stability_provider = TracerProvider()
stability_provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter(DB_PATH)))
tracer = stability_provider.get_tracer("llm-zoomcamp")

rag_stability = RAGTraced(index=index, llm_client=client)

for i in range(4):
    rag_stability.rag(query)
    print(f"run {i + 1}/4 done")

run 1/4 done
run 2/4 done
run 3/4 done
run 4/4 done


In [11]:
conn_stability = sqlite3.connect(DB_PATH)
df_stability = pd.read_sql("SELECT * FROM spans", conn_stability)

llm_tokens = df_stability[df_stability["name"] == "llm"]["input_tokens"]
print(llm_tokens.tolist())
print(f"min={llm_tokens.min()}, max={llm_tokens.max()}, mean={llm_tokens.mean():.1f}")

[7111.0, 7111.0, 7111.0, 7111.0]
min=7111.0, max=7111.0, mean=7111.0


**Answer Q6: `They're identical`**

All four runs came back with exactly the same `input_tokens` count. `minsearch`'s text search is deterministic for a fixed query against a fixed index, so it returns the same top-5 chunks every time, the same context, the same input token count. `output_tokens` (not shown) does vary a bit run to run, since that side depends on the model's own generation, but that's not what this question asks about.

## Summary of answers

| Question | Answer |
|---|---|
| Q1. Spans per trace | `3` |
| Q2. Input tokens | `7000` |
| Q3. LLM call duration | `Over 2000ms` |
| Q4. Span names in `spans` table | `rag`, `search`, and `llm` |
| Q5. Span with most total duration | `llm` |
| Q6. Input token variation across 4 runs | `They're identical` |

Submit at: https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw5